In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate
import pandas as pd

from src.custom_functions import get_parameters_files, MSE
from src.data_loader import load_data
from src.data_manipulation import convert_model_to_data_shape#convert_num_m_to_num_d, 

import glob, pdb

densities = np.arange(10,22,2,dtype=int)  
x_d = np.arange(75, 1875.1, 50)
x_m = np.arange(62.5, 1900,  25)

fontsize=18
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

true_model_names = {0:"rm_rp",1:"rm_rp",2:"rm_rp",3:"rm_rp",
                   4:"rm_pint_rp",5:"rm_rp_a0_a1",6:"rm_rp_a0_a1",7:"rm_pint_rp_a0_a1"}

In [3]:
## Artificial Data

data_type = "artificial"
pred_type = "fit_all"
simulations = np.arange(8)

formatted_parameter_names = {"rm_rp":"Model 1",
                             "rm_pint_rp":"Model 2",
                             "rm_rp_a0_a1":"Model 3",
                             "rm_pint_rp_a0_a1":"Model 4"}

for estimate_type in ["mean","MAP"]:

    headers = [""] + ["Simulation " + str(i+1) for i in range(8)]

    table = []
    table_ICs = []
    
    for model_type in ["PDE","ABM"]:
        for parameter_names in ["rm_rp","rm_pint_rp","rm_rp_a0_a1","rm_pint_rp_a0_a1"]:

            table_tmp =  [model_type + " "+ formatted_parameter_names[parameter_names]]
            table_ICs_tmp =  []

            for sim in simulations:
                
                #Load in simulation information
                input_parameters = [None, parameter_names, model_type, data_type, sim, pred_type]

                (parameter_names, 
                model_type, 
                data_type, 
                dens, 
                simulation,
                initial_perc,
                trueParams,
                delay, 
                lower_bounds,
                upper_bounds,
                indepParams,
                dxs,
                prediction_type,
                priorDir, 
                histogramDir, 
                posteriorDir) = get_parameters_files(input_parameters)

                #Load in data
                Data = load_data(data_type, dens=dens, simulation=simulation)

                #load in parameter estimate (mean or MAP)
                mat = np.load(f"{posteriorDir}mean_info.npy",allow_pickle=True).item()
                model_prediction = mat[f"{estimate_type}_prediction"]

                #Convert ABM data to data grid
                if model_type == "ABM":
                    model_prediction = convert_model_to_data_shape(np.mean(model_prediction,axis=0))

                ### Calculate AIC value
                
                #number of data points
                N = Data.size

                MSE_val = MSE(model_prediction, Data)
                paramNums = parameter_names.count("_") + 1

                ## AIC
                AIC = N*np.log(MSE_val) + 2*(paramNums+1)
                ### BIC
                # AIC = N*np.log(MSE_val) + np.log(N)*(paramNums+1)
                # ### CAIC
                # AIC = N*np.log(MSE_val) + (np.log(N)+1)*(paramNums+1)

                #to shade the cell
                if model_type == "ABM" and parameter_names == true_model_names[sim]:
                    table_tmp.append("\cellcolor{blue!25}" + f"{round(AIC,1)} ({round(MSE_val,2)})")
                elif model_type == "PDE" and parameter_names == true_model_names[sim]:
                    table_tmp.append("\cellcolor{green!25}" + f"{round(AIC,1)} ({round(MSE_val,2)})")    
                else:
                    table_tmp.append(f"{round(AIC,1)} ({round(MSE_val,2)})")
                table_ICs_tmp.append(AIC)
                    
                
            table.append(table_tmp)
            table_ICs.append(table_ICs_tmp)

    table_ICs = np.array(table_ICs)
    
    #Determine lowest, second-lowest AICs in each column
    lowest_indices = np.argsort(table_ICs, axis=0)[:2, :]

    for i,low_ind in enumerate(lowest_indices[0,:]):
        table[low_ind][i+1] = r"\textbf{" + table[low_ind][i+1] + "}"
    for i,low_ind in enumerate(lowest_indices[1,:]):
        table[low_ind][i+1] = r"\uline{" + table[low_ind][i+1] + "}"

    
    table_df = pd.DataFrame(data=table,columns=headers)

    latex_table = table_df.to_latex(
        index=False,  # To not include the DataFrame index as a column in the table
        position="h",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
        column_format="|l|" + "c"*len(simulations) + "|",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
        escape=False,  # Disable escaping LaTeX special characters in the DataFrame
        float_format="{:0.2f}".format  # Formats floats to two decimal places
    )

    print(latex_table)        

\begin{table}[h]
\begin{tabular}{|l|cccccccc|}
\toprule
 & Simulation 1 & Simulation 2 & Simulation 3 & Simulation 4 & Simulation 5 & Simulation 6 & Simulation 7 & Simulation 8 \\
\midrule
PDE Model 1 & \textbf{\cellcolor{green!25}220.0 (3.18)} & \textbf{\cellcolor{green!25}437.7 (10.31)} & \textbf{\cellcolor{green!25}285.7 (4.54)} & \textbf{\cellcolor{green!25}327.1 (5.67)} & \textbf{279.6 (4.39)} & 413.0 (9.02) & \uline{594.0 (24.01)} & 512.4 (15.45) \\
PDE Model 2 & \uline{226.6 (3.26)} & 438.9 (10.27) & \uline{292.5 (4.65)} & 342.5 (6.1) & \uline{\cellcolor{green!25}281.6 (4.39)} & 417.9 (9.17) & 604.7 (25.16) & 497.0 (14.06) \\
PDE Model 3 & 471.3 (12.11) & 447.0 (10.61) & 294.9 (4.66) & \uline{330.3 (5.65)} & 283.5 (4.39) & \textbf{\cellcolor{green!25}324.7 (5.48)} & \cellcolor{green!25}596.5 (23.82) & \uline{443.0 (10.38)} \\
PDE Model 4 & 228.7 (3.23) & 447.7 (10.54) & 296.4 (4.65) & 359.5 (6.54) & 291.8 (4.54) & 443.0 (10.27) & \textbf{414.6 (8.81)} & \textbf{\cellcolor{green!

In [4]:
#Wound healing data

data_type = "WH"
density_ids = np.arange(6)
densities = np.arange(10,22,2)
pred_type = "fit_all"

for estimate_type in ["mean", "MAP"]:

    headers = [estimate_type] + ["Density " + str(dens) for dens in densities]

    table = []
    table_ICs = []
    
    for model_type in ["PDE","ABM"]:
        for parameter_names in ["rm_rp","rm_pint_rp","rm_rp_a0_a1","rm_pint_rp_a0_a1"]:

            table_tmp =  [model_type + " "+ formatted_parameter_names[parameter_names]]
            table_ICs_tmp = []
            
            for dens_id in density_ids:
                
                #Load in simulation information
                input_parameters = [None, parameter_names, model_type, data_type, dens_id, pred_type]

                (parameter_names, 
                    model_type, 
                    data_type, 
                    dens, 
                    simulation,
                    initial_perc,
                    trueParams,
                    delay, 
                    lower_bounds,
                    upper_bounds,
                    indepParams,
                    dxs,
                    prediction_type,
                    priorDir, 
                    histogramDir, 
                    posteriorDir) = get_parameters_files(input_parameters)

                #Load in data
                Data = load_data(data_type, dens=dens, simulation=simulation)
                
                #load in parameter estimate (mean or MAP)
                mat = np.load(f"{posteriorDir}mean_info.npy",allow_pickle=True).item()
                model_prediction = mat[f"{estimate_type}_prediction"]

                #Convert ABM data to data grid
                if model_type == "ABM":
                    model_prediction = convert_model_to_data_shape(np.mean(model_prediction,axis=0))

                ### Calculate AIC value
                
                #number of data points
                N = Data.size

                MSE_val = MSE(model_prediction, Data)
                paramNums = parameter_names.count("_") + 1

                ## AIC
                AIC = N*np.log(MSE_val) + 2*(paramNums+1)
                ### BIC
                # AIC = N*np.log(MSE_val) + np.log(N)*(paramNums+1)
                # ### CAIC
                # AIC = N*np.log(MSE_val) + (np.log(N)+1)*(paramNums+1)
                
                table_tmp.append(f"{round(AIC,1)} ({round(MSE_val,2)})")
                table_ICs_tmp.append(AIC)
                
            table.append(table_tmp)
            table_ICs.append(table_ICs_tmp)

    table_ICs = np.array(table_ICs)
    
    #Determine lowest, second-lowest AICs in each column
    lowest_indices = np.argsort(table_ICs, axis=0)[:2, :]

    for i,low_ind in enumerate(lowest_indices[0,:]):
        table[low_ind][i+1] = r"\textbf{" + table[low_ind][i+1] + "}"
    for i,low_ind in enumerate(lowest_indices[1,:]):
        table[low_ind][i+1] = r"\uline{" + table[low_ind][i+1] + "}"

    
    table_df = pd.DataFrame(data=table,columns=headers)

    latex_table = table_df.to_latex(
        index=False,  # To not include the DataFrame index as a column in the table
        position="h",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
        column_format="|l|" + "c"*len(density_ids) + "|",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
        escape=False,  # Disable escaping LaTeX special characters in the DataFrame
        float_format="{:0.2f}".format  # Formats floats to two decimal places
    )

    print(latex_table)            

\begin{table}[h]
\begin{tabular}{|l|cccccc|}
\toprule
mean & Density 10 & Density 12 & Density 14 & Density 16 & Density 18 & Density 20 \\
\midrule
PDE Model 1 & 717.8 (46.87) & 660.3 (34.35) & 682.4 (38.72) & 684.2 (39.1) & 709.5 (44.81) & 773.5 (63.36) \\
PDE Model 2 & 716.7 (46.09) & 661.1 (34.13) & 659.4 (33.83) & 667.7 (35.38) & 705.7 (43.43) & 773.6 (62.7) \\
PDE Model 3 & 722.1 (46.95) & \uline{572.4 (20.91)} & \uline{657.3 (33.08)} & 601.7 (24.5) & 725.0 (47.69) & \uline{643.6 (30.71)} \\
PDE Model 4 & 842.4 (89.0) & \textbf{547.1 (18.04)} & \textbf{603.1 (24.41)} & \textbf{573.8 (20.84)} & \textbf{574.8 (20.95)} & \textbf{639.7 (29.76)} \\
ABM Model 1 & \textbf{700.8 (42.75)} & 650.1 (32.51) & 679.3 (38.07) & 678.5 (37.92) & 709.0 (44.69) & 767.9 (61.46) \\
ABM Model 2 & \uline{709.4 (44.32)} & 659.4 (33.82) & 662.1 (34.32) & 663.7 (34.61) & 707.2 (43.78) & 772.0 (62.16) \\
ABM Model 3 & 744.7 (53.06) & 978.1 (187.36) & 725.4 (47.8) & 755.3 (56.19) & 639.9 (30.12) & 651.0 (31

In [5]:
%load_ext watermark

%watermark -n -u -v -iv -w

Last updated: Sun Jun 07 2026

Python implementation: CPython
Python version       : 3.11.5
IPython version      : 8.15.0

pandas    : 2.1.4
matplotlib: 3.8.0
numpy     : 1.25.2

Watermark: 2.4.3

